In [ ]:

import random
import math

# Calculate number of attacking queen pairs
def cost(state):
    count = 0
    for i in range(8):
        for j in range(i + 1, 8):
            if state[i] == state[j] or abs(state[i] - state[j]) == abs(i - j):
                count += 1
    return count


# Generate a random neighbour
def neighbour(state):
    new = state.copy()
    col = random.randint(0, 7)
    new[col] = random.randint(0, 7)
    return new


# Simulated Annealing
def simulated_annealing():
    current = [random.randint(0, 7) for _ in range(8)]
    T = 100

    print("Initial State:", current)
    print("Initial Cost:", cost(current))

    while T > 0.001:
        if cost(current) == 0:
            break

        next_state = neighbour(current)

        delta_E = cost(current) - cost(next_state)

        if delta_E > 0:
            current = next_state
        else:
            p = math.exp(delta_E / T)
            if random.random() < p:
                current = next_state

        T = T * 0.99

    return current


# Display chessboard
def display(state):
    for i in range(8):
        for j in range(8):
            if state[j] == i:
                print("Q", end=" ")
            else:
                print(".", end=" ")
        print()


# Main Program
solution = simulated_annealing()

print("\nFinal State:", solution)
print("Final Cost:", cost(solution))
print("\nChessboard:")
display(solution)


In [ ]:

import random
import math
import matplotlib.pyplot as plt

# Calculate number of attacking queen pairs
def calculate_conflicts(state):
    conflicts = 0

    for i in range(8):
        for j in range(i + 1, 8):
            # Same row or same diagonal
            if state[i] == state[j] or \
               abs(state[i] - state[j]) == abs(i - j):
                conflicts += 1

    return conflicts


# Generate a random neighbouring state
def random_neighbour(state):
    new_state = state.copy()

    column = random.randint(0, 7)

    # Move queen to a different row
    available_rows = [
        r for r in range(8)
        if r != new_state[column]
    ]

    new_state[column] = random.choice(available_rows)

    return new_state


# Simulated Annealing Algorithm
def simulated_annealing(initial_state, temperature,
                        cooling_rate, max_iterations):

    current = initial_state.copy()
    current_cost = calculate_conflicts(current)

    best = current.copy()
    best_cost = current_cost

    print("\n========== ITERATION DETAILS ==========")
    print("Iteration | Temperature | Conflicts")

    for iteration in range(1, max_iterations + 1):

        if current_cost == 0:
            break

        if temperature < 0.0001:
            break

        next_state = random_neighbour(current)
        next_cost = calculate_conflicts(next_state)

        # Delta E = Current Cost - Next Cost
        delta_e = current_cost - next_cost

        if delta_e > 0:
            current = next_state
            current_cost = next_cost

        else:
            probability = math.exp(delta_e / temperature)

            if random.random() < probability:
                current = next_state
                current_cost = next_cost

        if current_cost < best_cost:
            best = current.copy()
            best_cost = current_cost

        if iteration <= 20 or iteration % 500 == 0:
            print(f"{iteration:^9} | "
                  f"{temperature:^11.4f} | "
                  f"{current_cost:^9}")

        temperature *= cooling_rate

    return best, iteration


# Display chessboard in terminal
def print_board(state):

    print("\n   0 1 2 3 4 5 6 7")

    for row in range(8):
        print(row, end="  ")

        for col in range(8):
            if state[col] == row:
                print("Q", end=" ")
            else:
                print(".", end=" ")

        print()


# Display graphical chessboard
def draw_board(state, title, filename):

    fig, ax = plt.subplots(figsize=(7, 7))

    for row in range(8):
        for col in range(8):

            color = "white" if (row + col) % 2 == 0 \
                    else "lightgray"

            ax.add_patch(
                plt.Rectangle(
                    (col, row), 1, 1,
                    facecolor=color,
                    edgecolor="black"
                )
            )

            if state[col] == row:
                ax.text(
                    col + 0.5,
                    row + 0.5,
                    "Q",
                    fontsize=24,
                    fontweight="bold",
                    color="darkred",
                    ha="center",
                    va="center"
                )

    ax.set_xlim(0, 8)
    ax.set_ylim(8, 0)
    ax.set_xticks(range(8))
    ax.set_yticks(range(8))
    ax.set_aspect("equal")

    ax.set_title(title, fontsize=15)
    plt.savefig(filename, bbox_inches="tight")
    plt.show()


# MAIN PROGRAM
print("========== SIMULATED ANNEALING ==========")
print("========== 8 QUEENS PROBLEM ==========")

print("\n1. Enter your own initial state")
print("2. Generate random initial state")

choice = input("\nEnter choice (1 or 2): ").strip()

if choice == "1":

    print("\nEnter 8 row positions (0 to 7).")
    print("One queen is placed in each column.")
    print("Example: 0 1 2 3 4 5 6 7")

    while True:
        try:
            state = list(map(
                int,
                input("Enter positions: ").split()
            ))

            if len(state) == 8 and \
               all(0 <= x <= 7 for x in state):
                break

            print("Enter exactly 8 numbers from 0 to 7.")

        except ValueError:
            print("Invalid input! Enter integers only.")

else:
    state = [random.randint(0, 7) for _ in range(8)]

print("\nEnter algorithm parameters.")
print("Press Enter to use the default values.")

temp_input = input("Initial temperature (default 100): ")
rate_input = input("Cooling rate (default 0.999): ")
iter_input = input("Max iterations (default 20000): ")

temperature = float(temp_input) if temp_input else 100.0
cooling_rate = float(rate_input) if rate_input else 0.999
max_iterations = int(iter_input) if iter_input else 20000

if temperature <= 0:
    raise ValueError("Temperature must be positive.")

if not (0 < cooling_rate < 1):
    raise ValueError("Cooling rate must be between 0 and 1.")

if max_iterations <= 0:
    raise ValueError("Max iterations must be positive.")


# Initial state
print("\n========== INITIAL STATE ==========")

print("Queen Positions:", state)
print("Initial Conflicts:", calculate_conflicts(state))

print_board(state)

draw_board(
    state,
    "Initial 8-Queens Configuration",
    "8_queens_initial.png"
)


# Run Simulated Annealing
solution, iterations = simulated_annealing(
    state,
    temperature,
    cooling_rate,
    max_iterations
)


# Final output
print("\n========== FINAL RESULT ==========")

print("Final Queen Positions:", solution)
print("Final Conflicts:", calculate_conflicts(solution))
print("Total Iterations:", iterations)

print_board(solution)

draw_board(
    solution,
    "Final 8-Queens Configuration",
    "8_queens_solution.png"
)

if calculate_conflicts(solution) == 0:
    print("\nSUCCESS! 8-Queens problem solved.")
    print("No two queens attack each other.")
else:
    print("\nNo perfect solution found in this run.")
    print("Try a higher temperature or slower cooling rate.")

print("\nImages saved successfully.")
